In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("email_data.csv")

In [3]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df.tail()

,Category,Message
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [5]:
df.isnull().sum()

Category    0
Message     0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(415)

In [ ]:
df.drop_duplicates(inplace=True)

In [11]:
df.shape

(5157, 2)

In [12]:
df.groupby("Category").size()

Category
ham     4516
spam     641
dtype: int64

In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 5157 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Category  5157 non-null   str  
 1   Message   5157 non-null   str  
dtypes: str(2)
memory usage: 120.9 KB


In [15]:
df['Category'] = df['Category'].map({'ham': 0, 'spam': 1})
df.head()


,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [19]:
from sklearn.model_selection import train_test_split
X = df["Message"]
y = df["Category"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=42, stratify=df["Category"])

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4125,)
Testing data: (1032,)


In [ ]:
import re
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download("stopwords")
nltk.download("punkt_tab")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = word_tokenize(text)
    cleaned_words = [lemmatizer.lemmatize(i) for i in words if i not in stop_words]
    return " ".join(cleaned_words)

X_train_cleaned = X_train.apply(clean_text)
X_test_cleaned = X_test.apply(clean_text)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train_cleaned)
X_test_vec = vectorizer.transform(X_test_cleaned)

In [21]:
print(type(X_train_vec))

<class 'scipy.sparse._csr.csr_matrix'>


In [22]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train_vec, y_train)
print("Model Training complete.")

Model Training complete.


In [23]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Model Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

Overall Model Accuracy: 96.22%

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       904
           1       1.00      0.70      0.82       128

    accuracy                           0.96      1032
   macro avg       0.98      0.85      0.90      1032
weighted avg       0.96      0.96      0.96      1032

CONFUSION MATRIX
[[904   0]
 [ 39  89]]


In [24]:
def predict_custom_email(email_text):
    cleaned_input = clean_text(email_text)
    vectorized_input = vectorizer.transform([cleaned_input])
    prediction = model.predict(vectorized_input)
    probabilities = model.predict_proba(vectorized_input)[0]
    ham_prob = probabilities[0] * 100
    spam_prob = probabilities[1] * 100
    result_label = "SPAM" if prediction == 1 else "HAM"
    print(f"Input Text: \"{email_text}\"")
    print(f"Classification Result: {result_label}")
    print(f"Confidence Metrics -> Ham: {ham_prob:.2f}% | Spam: {spam_prob:.2f}%")
    print("-" * 60)

print("-------LIVE MODEL INFERENCE TESTING-------\n")

predict_custom_email("Hey, are we still meeting up for coffee and lunch at 2pm today?")

predict_custom_email("URGENT! Click this link to claim your free cash prize lottery now?")

predict_custom_email("Dear customer, please review the attached invoice regarding your recent account transaction.")


-------LIVE MODEL INFERENCE TESTING-------

Input Text: "Hey, are we still meeting up for coffee and lunch at 2pm today?"
Classification Result: HAM
Confidence Metrics -> Ham: 99.68% | Spam: 0.32%
------------------------------------------------------------
Input Text: "URGENT! Click this link to claim your free cash prize lottery now?"
Classification Result: SPAM
Confidence Metrics -> Ham: 4.20% | Spam: 95.80%
------------------------------------------------------------
Input Text: "Dear customer, please review the attached invoice regarding your recent account transaction."
Classification Result: HAM
Confidence Metrics -> Ham: 68.02% | Spam: 31.98%
------------------------------------------------------------
